# Survey speed vs. image quality: static PNG

Matplotlib version of the figure built in `04_interactive.ipynb`, from the same cached tables.
6 x 4 in, 12 pt axis labels, `TRANSPARENT` background switch (set to white while iterating; flip
back to transparent for delivery), Rubin colour palette (`ohlp/style.py`).

`PERIOD = "early_ops"` draws a **single summary point** for Early Ops, 24 Oct–30 Nov 2025 (the RASAB 2026 slide),
i.e. the nights before the Dec 2025–Mar 2026 sequence of downtimes (water condensation, Dynalene loss,
vacuum leak, earthquake; see `nightly_survey_speed/plots/downtime_summary.md`), with its nights as context
and the window described in the legend at 8 pt, as in the three-period figure.
`PERIOD = "early_ops_vs_lsst"` adds a **second point** for the LSST on-sky stretch, 29 Jun–14 Jul 2026
(the range of Lynne Jones's `diq_vs_visit_rate` plot; survey visits end on 13 Jul, 14 Jul is a closed weather night), in purple against the Early Ops teal,
with both nightly clouds. Each window is described in the legend
(`LABELS_IN_LEGEND`); set it to `False` for labels next to the points instead.
`PERIOD = "three_periods"` draws **three points on one figure** (Bruno's RASAB slide, 2026-10-22): the first
three months of Early Ops (24 Oct 2025 – 23 Jan 2026, teal circle), LSST from its start to the last night
on sky (29 Jun – 13 Jul 2026, purple square) and LSST from its start to `DAY_OBS_MAX` (today; hollow purple
square, the storm nights since 14 Jul 2026 counted as zeros). All nights are drawn as context, each once;
the legend describes each window at `LEGEND_SIZE` pt with "n of m nights w/ on sky data". No arrow joins
the two LSST points because their windows overlap.
`SHOW_STAR = False` omits the orange "LSST 10 year Average" star (file suffix `_nostar`).


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

import pandas as pd

warnings.simplefilter("ignore", FutureWarning)
sys.path.insert(0, ".")
from ohlp import binning, data, static, reference as ref
from ohlp.style import rp

In [ ]:
# Parameters (must match 01_data_and_binning.ipynb)
DAY_OBS_MIN = ref.DAY_OBS_EARLY_OPS_START
DAY_OBS_MAX = 20261022
PERIOD = "three_periods"  # "early_ops" (single Oct+Nov 2025 point), "early_ops_vs_lsst" (adds the LSST point), "three_periods" (Early Ops 3 months + LSST on sky + LSST to date), "window", "quarter" or "month"
IQ_COL = "idiq_aos_cam"
FIGSIZE = (6, 4)  # inches
LABEL_SIZE = 12  # pt, axis labels
TRANSPARENT = True  # False = white background (easier on a dark screen); True for delivery
TITLE = None  # None for no title; set per PERIOD below
LEGEND_SIZE = None  # pt, period legend; None = text size (9); "early_ops" and "three_periods" use 8
NIGHTS_SUFFIX = ""  # appended to the "n of m nights" legend line; "early_ops" and "three_periods" use " w/ on sky data"
HOLLOW = False  # outline-only period markers (bool, or one flag per period); set per PERIOD below
SHOW_STAR = True  # False omits the orange "LSST 10 year Average" star; the file gets a "_nostar" suffix

# Single Early Ops window: first survey night to the last night before the downtime sequence.
EARLY_OPS_WINDOW = {"Early Ops": (ref.DAY_OBS_EARLY_OPS_START, 20251130)}  # the legend adds the date span
# LSST on sky: season start to 20260714, the same range as Lynne Jones's diq_vs_visit_rate plot
# (20260629 to 20260714) so the two figures agree. 20260714 is a closed weather night: the last
# survey visits are on 20260713 and the storm closes the dome from 20260714.
LSST_WINDOW = {"LSST": (ref.DAY_OBS_LSST_START, ref.DAY_OBS_LAST_BEFORE_STORM)}
# Three periods on one figure. Every calendar night of a window is counted, so the "to date" window
# includes the storm closure (no survey visits after 20260713) as zeros.
THREE_PERIODS = {
    "Early Ops": (ref.DAY_OBS_EARLY_OPS_START, 20260123),  # first three months (92 nights)
    "LSST, on sky": (ref.DAY_OBS_LSST_START, ref.DAY_OBS_LAST_SURVEY_VISIT),
    "LSST, to date": (ref.DAY_OBS_LSST_START, DAY_OBS_MAX),
}
THREE_PERIODS_COLORS = [rp.TEAL, rp.CATEGORICAL[2], rp.CATEGORICAL[2]]  # teal, purple, purple
THREE_PERIODS_HOLLOW = [False, False, True]  # the open-ended "to date" window is drawn as an outline

tag = f"{DAY_OBS_MIN}_{DAY_OBS_MAX}"
nightly = pd.read_parquet(data.DATA_DIR / f"nightly_{tag}.parquet")
if PERIOD == "early_ops":
    (lo, hi), = EARLY_OPS_WINDOW.values()
    visits = pd.read_parquet(data.DATA_DIR / f"visits_{tag}.parquet")
    periods = binning.window_table(visits, nightly, EARLY_OPS_WINDOW)
    stem = f"early_ops_octnov_{lo}-{hi}"
    periods.to_parquet(data.DATA_DIR / f"window_early_ops_octnov_{lo}_{hi}.parquet")
    periods.to_csv(data.DATA_DIR / f"window_early_ops_octnov_{lo}_{hi}.csv", index=False)
    TITLE = TITLE or "Early Ops, 24 Oct – 30 Nov 2025"
    LEGEND_SIZE = LEGEND_SIZE or 8
    NIGHTS_SUFFIX = NIGHTS_SUFFIX or " w/ on sky data"
elif PERIOD == "early_ops_vs_lsst":
    (lo, _), = EARLY_OPS_WINDOW.values()
    (_, hi), = LSST_WINDOW.values()
    visits = pd.read_parquet(data.DATA_DIR / f"visits_{tag}.parquet")
    periods = binning.window_table(visits, nightly, {**EARLY_OPS_WINDOW, **LSST_WINDOW})
    stem = f"early_ops_vs_lsst_{lo}-{hi}"
    periods.to_parquet(data.DATA_DIR / f"windows_early_ops_vs_lsst_{lo}_{hi}.parquet")
    periods.to_csv(data.DATA_DIR / f"windows_early_ops_vs_lsst_{lo}_{hi}.csv", index=False)
    TITLE = TITLE or "Early Ops (Oct–Nov 2025) vs. LSST (Jun–Jul 2026)"
    COLORS = [rp.TEAL, rp.CATEGORICAL[2]]  # teal, purple
    LABEL_OFFSETS = {0: (9, 9), 1: (-9, -14)}  # Early Ops above-right, LSST below-left
elif PERIOD == "three_periods":
    lo, hi = DAY_OBS_MIN, DAY_OBS_MAX
    visits = pd.read_parquet(data.DATA_DIR / f"visits_{tag}.parquet")
    periods = binning.window_table(visits, nightly, THREE_PERIODS)
    stem = f"three_periods_{lo}-{hi}"
    periods.to_parquet(data.DATA_DIR / f"windows_three_periods_{lo}_{hi}.parquet")
    periods.to_csv(data.DATA_DIR / f"windows_three_periods_{lo}_{hi}.csv", index=False)
    TITLE = TITLE or "Early Ops vs. LSST"
    COLORS = THREE_PERIODS_COLORS
    HOLLOW = THREE_PERIODS_HOLLOW
    LEGEND_SIZE = LEGEND_SIZE or 8
    NIGHTS_SUFFIX = NIGHTS_SUFFIX or " w/ on sky data"
else:
    TABLE_NAME = {"window": "windows", "quarter": "quarterly", "month": "monthly"}[PERIOD]
    periods = pd.read_parquet(data.DATA_DIR / f"{TABLE_NAME}_{tag}.parquet")
    stem = f"{TABLE_NAME}_{DAY_OBS_MIN}-{DAY_OBS_MAX}"
periods[["label_long", "n_nights_calendar", "n_nights_with_visits", "n_visits", "n_vis_expected", "ratio", f"{IQ_COL}_med"]].round(3)

In [ ]:
fig, ax = static.build_static_figure(
    periods,
    nightly,
    iq_col=IQ_COL,
    figsize=FIGSIZE,
    label_size=LABEL_SIZE,
    transparent=TRANSPARENT,
    title=TITLE,
    colors=COLORS,
    label_offsets=LABEL_OFFSETS,
    labels_in_legend=LABELS_IN_LEGEND and PERIOD in ("early_ops", "early_ops_vs_lsst", "three_periods"),
    legend_size=LEGEND_SIZE,
    nights_suffix=NIGHTS_SUFFIX,
    hollow=HOLLOW,
    show_star=SHOW_STAR,
)
out = static.save_png(
    fig,
    data.DATA_DIR.parent / "figures" / f"speed_vs_iq_static_{stem}{'' if SHOW_STAR else '_nostar'}.png",
    transparent=TRANSPARENT,
)
print(out)